# Lenguajes y Autómatas II (SCD-1016)
## Sesión 33 · Derramamiento a Memoria (Register Spilling)

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Qué hace el compilador cuando tiene más valores vivos que registros disponibles?
>
> **Objetivo de la sesión:** Calcular cuántas ranuras de derrame necesita una función y elegir qué valor mandar a la pila cuando la presión de registros aprieta.

**Entregable:** tu implementación de ranuras_derrame() y victima_desalojo() en artesano_sesion33_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion33.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Recibes cuántos valores están vivos y cuántos registros hay, o el arreglo de próximos usos de cada valor; devuelves las ranuras necesarias o el índice del valor que debe irse a la pila.

Al final de la sesión tendrás: **Dos funciones: una que dimensiona el área de derrame y otra que elige la víctima correcta según su próximo uso.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 33](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion33.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion33.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion33.h"

int ranuras_derrame(int vivas, int registros);
int victima_desalojo(const int *proximos, int n);
```

- Si los valores vivos caben en los registros, no se reserva ninguna ranura.
- La víctima del desalojo es el valor con el próximo uso más lejano; ante empate gana el índice menor.
- Un arreglo vacío o NULL no tiene víctima: se responde con -1.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion33_c.c` junto con `artesano_sesion33_c.c`
> (o `ia_sesion33_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `ranuras_derrame(12, 8)` | 4 |
| 2 | `ranuras_derrame(10, 8)` | 2 |
| 3 | `ranuras_derrame(3, 8)` | 0 |
| 4 | `victima_desalojo({1,9,4}, 3)` | 1 |
| 5 | `victima_desalojo({2,2,2}, 3)` | 0 |
| 6 | `victima_desalojo(NULL, 0)` | -1 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 33 - Manejo de pila ante agotamiento de registros ===
Caso                               esperado:              obtenido:              resultado
[ 1] ranuras_derrame(12, 8)             esperado: 4            obtenido: 4            OK
[ 2] ranuras_derrame(10, 8)             esperado: 2            obtenido: 2            OK
[ 3] ranuras_derrame(3, 8)              esperado: 0            obtenido: 0            OK
[ 4] victima_desalojo({1,9,4}, 3)       esperado: 1            obtenido: 1            OK
[ 5] victima_desalojo({2,2,2}, 3)       esperado: 0            obtenido: 0            OK
[ 6] victima_desalojo(NULL, 0)          esperado: -1           obtenido: -1           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion33.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] ranuras_derrame nunca reserva de más ni de menos: solo cuenta los valores que no caben.
- [ ] victima_desalojo aplica la heurística del próximo uso más lejano y resuelve el empate por índice.
- [ ] Los seis casos del harness salen OK y el programa termina con código de salida 0.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion33_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion33.h
/* api_sesion33.h - contrato de la Sesion 33
 * Manejo de pila ante agotamiento de registros
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION33_H
#define API_SESION33_H

int ranuras_derrame(int vivas, int registros);
int victima_desalojo(const int *proximos, int n);

#endif /* API_SESION33_H */


In [ ]:
%%writefile artesano_sesion33_c.c
/* artesano_sesion33_c.c - Fase 1: El Artesano (sin IA)
 * Manejo de pila ante agotamiento de registros.
 *
 * Implementa lo que declara api_sesion33.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion33.h"
#include <stdio.h>
#include <string.h>

int ranuras_derrame(int vivas, int registros) {
    (void)vivas;
    (void)registros; /* TODO: solo sobran los que no caben */
    return 0;
}

int victima_desalojo(const int *proximos, int n) {
    (void)proximos;
    (void)n; /* TODO: el proximo uso mas lejano */
    return 0;
}


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion33_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion33_c.c
/* ia_sesion33_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion33.h. No puede traer main().
 */
#include "api_sesion33.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion33_c.c
/* harness_sesion33_c.c - auditoria automatica de la Sesion 33
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion33_c.c artesano_sesion33_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion33.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 33 - Manejo de pila ante agotamiento de registros ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { int r = ranuras_derrame(12, 8); verificar_int("ranuras_derrame(12, 8)", (long)(4), (long)(r)); }
    { int r = ranuras_derrame(10, 8); verificar_int("ranuras_derrame(10, 8)", (long)(2), (long)(r)); }
    { int r = ranuras_derrame(3, 8); verificar_int("ranuras_derrame(3, 8)", (long)(0), (long)(r)); }
    { int v[] = {1, 9, 4}; int r = victima_desalojo(v, 3); verificar_int("victima_desalojo({1,9,4}, 3)", (long)(1), (long)(r)); }
    { int v[] = {2, 2, 2}; int r = victima_desalojo(v, 3); verificar_int("victima_desalojo({2,2,2}, 3)", (long)(0), (long)(r)); }
    { int r = victima_desalojo(NULL, 0); verificar_int("victima_desalojo(NULL, 0)", (long)(-1), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion33.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion33_c.c artesano_sesion33_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion33_c.c ia_sesion33_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion33.md` a `auditorias/auditoria_sesion33.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
